# Pipeline CoT penuh — A6000 (48 GB)

Input: `data/Final/easy_clean_v4.jsonl` + `numglue_clean_v4.jsonl` (mundur otomatis ke `_v3`
kalau v4 belum ada). Kolom `cara` sudah dibuang — seluruh CoT datang dari teacher model.

Urutan: muat data → holdout + train pool (pakai ulang hasil #4 kalau ada) → **bake-off teacher**
→ generate penuh dengan pemenang → rejection sampling → ChatML → **verifikasi pasangan soal** →
**statistik panjang token** → paket.

Beda dari notebook Kaggle: single GPU (`tensor_parallel_size=1`), `bfloat16` (Ampere punya bf16,
T4 tidak), tanpa workaround flashinfer, dan `aimo_hard` tidak dipakai.

**Jalankan berurutan.** Setiap tahap berat sudah resumable — kalau sesi mati, jalankan ulang selnya
dan ia melanjutkan dari checkpoint terakhir.

Sebelum melepas run 4–8 jam: uji sel 6–8 dulu dengan `limit=` kecil. Notebook ini sudah lolos
dry-run CPU tapi **belum teruji GPU**.


In [ ]:
# ── 1. Setup ───────────────────────────────────────────────────────────────
# A6000 = Ampere (sm_86): bf16 didukung, flashinfer tidak perlu di-uninstall
# (workaround itu khusus T4 compute 7.5 di notebook Kaggle).
!pip -q install "vllm>=0.6" langdetect sympy math-verify

import sys, os
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "src").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if not (REPO / "src").exists():
    !git clone -q https://github.com/henray404/FP_NLP.git
    REPO = Path("FP_NLP").resolve()
sys.path.insert(0, str(REPO))
os.chdir(REPO)

import torch
p = torch.cuda.get_device_properties(0)
print(f"REPO : {REPO}")
print(f"GPU  : {p.name}  {p.total_memory/1e9:.0f} GB  sm_{p.major}{p.minor}")
print(f"bf16 : {torch.cuda.is_bf16_supported()}")
assert torch.cuda.is_available(), "GPU tidak terdeteksi"

In [ ]:
# ── 2. Muat data terbaru (v4 kalau ada, kalau tidak v3) ────────────────────
# v4 = keluaran judge v4 (#3). Kalau branch #3 belum di-merge, notebook tetap jalan di v3;
# versi yang benar-benar dipakai dicetak supaya tercatat di komentar issue.
import json
from collections import Counter
from src.cot_synthesis.utils import read_jsonl

DATA = REPO / "data" / "Final"
BASENAMES = ["easy_clean", "numglue_clean"]   # aimo_hard TIDAK dipakai

subsets, VERSI = {}, {}
for base in BASENAMES:
    p4, p3 = DATA / f"{base}_v4.jsonl", DATA / f"{base}_v3.jsonl"
    p = p4 if p4.exists() else p3
    assert p.exists(), f"tidak ketemu {p4} maupun {p3} -- git pull / jalankan judge dulu"
    rows = read_jsonl(p)
    src = base.split("_")[0]
    for r in rows:
        r["source"] = src
    subsets[src] = rows
    VERSI[src] = p.name
    has_cara = sum("cara" in r for r in rows)
    print(f"{p.name:24} {len(rows):5} soal   baris masih punya 'cara': {has_cara}")
    assert has_cara == 0, "kolom `cara` masih ada -- pakai file v3/v4, bukan v2"

all_rows = [r for rows in subsets.values() for r in rows]
print(f"\nTOTAL: {len(all_rows)} soal   {Counter(r['source'] for r in all_rows)}")
print("versi terpakai:", VERSI)
print("contoh:", json.dumps(all_rows[0], ensure_ascii=False)[:160])


In [ ]:
# ── 3. Holdout + train pool (pakai ulang hasil #4 kalau sudah ada) ─────────
# #4 adalah GERBANG data beku. Kalau holdout.jsonl + train_pool.jsonl sudah ada, JANGAN
# di-split ulang -- split ulang mengubah komposisi holdout dan membuat angka #4 tidak
# sebanding. Split di bawah hanya jalan sebagai cadangan kalau file itu belum ada.
import random, re
from src.eval.make_holdout import answer_type, GRADEABLE

N_HOLDOUT, SEED = 300, 42
EVAL_DIR = REPO / "data" / "eval"; EVAL_DIR.mkdir(parents=True, exist_ok=True)
HOLDOUT_P, POOL_P = EVAL_DIR / "holdout.jsonl", DATA / "train_pool.jsonl"

def norm(s): return re.sub(r"\s+", " ", (s or "").strip().lower())

def dump(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return path

if HOLDOUT_P.exists() and POOL_P.exists():
    holdout, train_pool = read_jsonl(HOLDOUT_P), read_jsonl(POOL_P)
    print(f"PAKAI ULANG hasil #4: holdout={len(holdout)}  train_pool={len(train_pool)}")
    bocor = len({norm(r["soal"]) for r in train_pool} & {norm(r["soal"]) for r in holdout})
    print(f"cek ulang dekontaminasi: {bocor} soal train_pool bocor ke holdout "
          f"({'OK' if bocor == 0 else 'PERIKSA #4'})")
else:
    holdout, train_pool = [], []
    for src, rows in subsets.items():
        for r in rows:
            r["answer_type"] = answer_type(r.get("jawaban", ""))
        gradeable = [r for r in rows if r["answer_type"] in GRADEABLE]
        random.Random(SEED).shuffle(gradeable)
        picked = gradeable[:N_HOLDOUT]
        picked_ids = {id(r) for r in picked}
        holdout += picked
        train_pool += [r for r in rows if id(r) not in picked_ids]
        print(f"{src:8} gradeable={len(gradeable):5}  holdout={len(picked)}  "
              f"pool={len(rows)-len(picked)}")

    hold_keys = {norm(r["soal"]) for r in holdout}
    before = len(train_pool)
    train_pool = [r for r in train_pool if norm(r["soal"]) not in hold_keys]
    print(f"\ndekontaminasi: {before - len(train_pool)} baris latih dibuang (bocor ke holdout)")
    dump(holdout, HOLDOUT_P); dump(train_pool, POOL_P)

print(f"holdout    : {len(holdout)}  -> {HOLDOUT_P}")
print(f"train_pool : {len(train_pool)}  -> {POOL_P}")
print("komposisi holdout:", Counter(r["source"] for r in holdout))


In [ ]:
# ── 4. Kandidat teacher ────────────────────────────────────────────────────
# Semua muat bf16 di 48 GB, single GPU. Komentari yang tidak dipakai untuk hemat waktu.
#
# TIER 14B -- perbandingan terkontrol: parameter sama, 3 filosofi training berbeda.
# Ini inti eksperimennya; kalau waktu mepet, jalankan tier ini saja.
CANDIDATES = {
    "qwen3-14b":      "Qwen/Qwen3-14B",                             # ~28 GB  generalis multibahasa
    "r1-distill-14b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",   # ~28 GB  distilasi reasoning
    "openmath-14b":   "nvidia/OpenMath-Nemotron-14B",               # ~28 GB  spesialis matematika

    # TIER 7B -- kontinuitas dengan Skenario 1 paper lama
    "r1-distill-7b":  "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",    # ~15 GB  juara paper lama
    "qwen25-math-7b": "Qwen/Qwen2.5-Math-7B-Instruct",              # ~15 GB  runner-up paper lama
}

# Cadangan (butuh kuantisasi, 32B bf16 = ~64 GB > 48 GB):
#   nvidia/OpenMath-Nemotron-32B, Qwen/Qwen3-32B  -> pakai varian AWQ/GPTQ
# Cadangan model AIMO-2 juara Kaggle (subset OpenMathReasoning):
#   nvidia/OpenMath-Nemotron-14B-Kaggle
# Cadangan murah:
#   Qwen/Qwen3-8B (~16 GB), Qwen/Qwen3-4B (~8 GB), nvidia/AceMath-RL-Nemotron-7B (~15 GB)

est = {"14b": 28, "7b": 15}
for tag, m in CANDIDATES.items():
    print(f"{tag:16} {m}")
print(f"\n{len(CANDIDATES)} kandidat x {150} soal x {4} sampel = "
      f"{len(CANDIDATES)*150*4} generasi")

## Bake-off teacher

Paper lama memakai `DeepSeek-R1-Distill-Qwen-7B` (retensi 37,08%, cakupan 66,61%) karena T4 16 GB
tidak muat lebih besar. A6000 48 GB membuka pilihan, jadi teacher dipilih ulang secara empiris.

**Desain: tier 14B berisi tiga filosofi training pada jumlah parameter yang sama**, sehingga yang
diisolasi adalah keluarga model, bukan kapasitas:

| kandidat | basis | filosofi |
|---|---|---|
| `Qwen3-14B` | Qwen3 | generalis, multibahasa native (klaim 119 bahasa incl. Indonesia) |
| `DeepSeek-R1-Distill-Qwen-14B` | Qwen2.5-14B | distilasi jejak reasoning R1 |
| `OpenMath-Nemotron-14B` | Qwen2.5-14B | spesialis matematika, fine-tune OpenMathReasoning |

`OpenMath-Nemotron` penting secara metodologis: ia **berasal dari solusi juara AIMO-2**
(arXiv:2504.16891) — rujukan [5] paper ini, yaitu pendekatan yang sedang direplikasi. Memakai
teacher dari karya asalnya membuat klaim replikasi jauh lebih kuat.

### Hipotesis yang diuji (bukan sekadar cari model terbaik)

**H1 — spesialis matematika akan gagal di format/bahasa Indonesia.** Bukti dari paper ini sendiri:
Tabel XI/XII mencatat `OpenMath-Nemotron-1.5B` punya kepatuhan format hanya **0,147** (numglue) dan
**0,219** (easy) — terburuk dari semua model yang diuji. Kandidat tanpa `\boxed{}` langsung
dieliminasi `filter_solutions.py`, jadi kelemahan itu langsung memotong retensi.

**H2 — R1-Distill akan paling parah language-mixing-nya.** Literatur melaporkan model R1-Distill
justru **memperbesar** language-mixing dibanding backbone-nya ketika input bukan Inggris/Mandarin
(arXiv:2505.14815). Paper ini sudah mencatat gejalanya: ~58% langkah hasil pengisian tercampur
Inggris.

Kalau H1 dan H2 benar, pemenangnya `Qwen3-14B` — dan itu temuan yang layak dilaporkan: **untuk
distilasi CoT non-Inggris, generalis multibahasa mengalahkan spesialis matematika**, karena akurasi
matematis percuma kalau keluarannya terbuang di filter bahasa/format.

### Metrik

Selain retensi dan cakupan (seperti Skenario 1 lama), ditambah **rasio bahasa Indonesia** — metrik
yang paper belum pernah ukur. Alasannya: `to_chatml(id_only=True)` membuang CoT dominan Inggris,
jadi solusi benar tapi berbahasa Inggris **tetap terbuang**. Cakupan mentah menyesatkan.

Skor pemilihan = cakupan × rasio Indonesia.

In [ ]:
# ── 5a. Bake-off: generate kandidat, satu model per kali ───────────────────
import gc, torch
from src.cot_synthesis.generate import run_generate

COT = REPO / "data" / "cot"; COT.mkdir(parents=True, exist_ok=True)

BAKEOFF_N = 150   # soal per kandidat (cukup untuk membedakan, murah)
N_SAMPLES = 4     # kandidat solusi per soal saat bake-off (full run pakai 8)
MAX_TOKENS = 8192 # model thinking butuh ruang; R1/Qwen3 bisa panjang
BATCH = 64
DTYPE = "bfloat16"

def free_gpu():
    """Bebaskan VRAM antar model. vLLM tidak selalu melepas sendiri."""
    gc.collect()
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel
        destroy_model_parallel()
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print(f"VRAM terpakai: {torch.cuda.memory_allocated()/1e9:.1f} GB")

for tag, model in CANDIDATES.items():
    out = COT / f"bakeoff_{tag}.jsonl"
    print(f"\n{'='*60}\n{tag}  ->  {model}\n{'='*60}", flush=True)
    free_gpu()
    r = run_generate(
        DATA / "train_pool.jsonl", out,
        backend="vllm", model=model,
        n=N_SAMPLES, temperature=0.7, top_p=0.95,
        max_tokens=MAX_TOKENS, limit=BAKEOFF_N,
        tensor_parallel_size=1, batch_size=BATCH, dtype=DTYPE,
    )
    print(r)
    free_gpu()

In [ ]:
# ── 5b. Nilai kandidat bake-off: retensi, cakupan, format, RASIO INDONESIA ─
# Judge dimuat sekali untuk semua kandidat (hemat load model).
free_gpu()

from src.cot_synthesis.filter_solutions import run_filter
from src.cot_synthesis.to_chatml import is_indonesian
from src.cot_synthesis.utils import extract_boxed

bakeoff = {}
for tag, model in CANDIDATES.items():
    cand_p = COT / f"bakeoff_{tag}.jsonl"
    corr_p = COT / f"bakeoff_{tag}_correct.jsonl"
    if not cand_p.exists():
        print(f"lewati {tag}: {cand_p} tidak ada"); continue

    s = run_filter(cand_p, corr_p, judge_backend="vllm",
                   judge_model="Qwen/Qwen2.5-7B-Instruct",
                   prefilter=True, batch_size=256, resume=True)

    cands = read_jsonl(cand_p)
    n = max(len(cands), 1)
    tried = len({r["id"] for r in cands})
    bakeoff[tag] = {
        "model": model,
        "kandidat": len(cands),
        "benar": s["kept"],
        "retensi_%": round(100 * s["kept"] / n, 2),
        "cakupan_%": round(100 * s["problems_covered"] / max(tried, 1), 2),
        # H1: spesialis matematika diduga gagal format \boxed{} pada prompt Indonesia
        "format_%": round(100 * sum(extract_boxed(r["text"]) is not None for r in cands) / n, 2),
        # H2: R1-Distill diduga paling parah language-mixing
        "indonesia_%": round(100 * sum(is_indonesian(r["text"]) for r in cands) / n, 2),
    }
    print(tag, bakeoff[tag], flush=True)

# Solusi benar tapi berbahasa Inggris akan dibuang to_chatml(id_only=True),
# jadi cakupan mentah menyesatkan. Skor efektif = cakupan x rasio Indonesia.
for m in bakeoff.values():
    m["skor_efektif"] = round(m["cakupan_%"] * m["indonesia_%"] / 100, 2)

print("\n| teacher | retensi% | cakupan% | format% | Indonesia% | skor efektif |")
print("|---|---|---|---|---|---|")
for tag, m in sorted(bakeoff.items(), key=lambda kv: -kv[1]["skor_efektif"]):
    print(f"| {tag} | {m['retensi_%']} | {m['cakupan_%']} | {m['format_%']} | "
          f"{m['indonesia_%']} | {m['skor_efektif']} |")

WINNER_TAG = max(bakeoff, key=lambda t: bakeoff[t]["skor_efektif"])
WINNER = CANDIDATES[WINNER_TAG]
(COT / "bakeoff_summary.json").write_text(
    json.dumps({"hasil": bakeoff, "pemenang": WINNER_TAG}, ensure_ascii=False, indent=2),
    encoding="utf-8")
print(f"\nPEMENANG: {WINNER_TAG}  ({WINNER})")

In [ ]:
# ── 6. Generate penuh dengan teacher pemenang ──────────────────────────────
# Set manual kalau mau override hasil bake-off:
# WINNER, WINNER_TAG = "Qwen/Qwen3-14B", "qwen3-14b"
print("teacher:", WINNER)

# n=8 = resep AIMO-2 (sama dengan paper). KALAU WAKTU MEPET: N_FULL = 4 -> waktu separuh,
# cakupan turun sedikit karena peluang minimal satu sampel benar per soal berkurang.
N_FULL = 8
print(f"{len(train_pool)} soal x {N_FULL} sampel = {len(train_pool) * N_FULL} generasi")

# candidates_<teacher>.jsonl SENGAJA disimpan: jadi bahan mentah #9 (union teacher) dan
# #10 (pasangan DPO/KTO) tanpa perlu generasi ulang. Jangan dihapus setelah run.
CAND_P = COT / f"candidates_{WINNER_TAG}.jsonl"

free_gpu()
res = run_generate(
    DATA / "train_pool.jsonl", CAND_P,
    backend="vllm", model=WINNER,
    n=N_FULL, temperature=0.7, top_p=0.95,
    max_tokens=MAX_TOKENS, dtype=DTYPE,
    tensor_parallel_size=1, batch_size=BATCH,
)
print(res)
print("kandidat mentah ->", CAND_P)


In [ ]:
# ── 7. Rejection sampling penuh (judge Qwen2.5-7B-Instruct) ────────────────
# Judge dimuat SETELAH teacher dibebaskan. Resumable via .progress.
free_gpu()

from src.cot_synthesis.filter_solutions import run_filter

# correct_<teacher>.jsonl juga SENGAJA disimpan -- dipakai #9 dan #10.
CORRECT_P = COT / f"correct_{WINNER_TAG}.jsonl"

stats = run_filter(
    CAND_P, CORRECT_P,
    judge_backend="vllm",
    judge_model="Qwen/Qwen2.5-7B-Instruct",
    prefilter=True,      # cek string/math_verify gratis dulu -> hemat panggilan judge
    batch_size=256,
    resume=True,
)
print(stats)
print(f"\nretensi : {100*stats['kept']/max(stats['total'],1):.2f}%")
print(f"cakupan : {100*stats['problems_covered']/len(train_pool):.2f}% "
      f"({stats['problems_covered']}/{len(train_pool)} soal)")
print("solusi benar ->", CORRECT_P)


In [ ]:
# ── 8. Bangun ChatML (cot.jsonl + nocot.jsonl) ─────────────────────────────
# MENIMPA data lama di data/sft/train (2.709 baris dari pipeline lama) -- disengaja.
from src.cot_synthesis.to_chatml import run as build_chatml

# data/sft/train/ -- path ini yang dibaca src/training/configs/*.yaml
SFT_DIR = REPO / "data" / "sft" / "train"
cm = build_chatml(CORRECT_P, SFT_DIR,
                  best_per_problem=True,   # 1 solusi terbaik per soal
                  id_only=True)            # buang CoT dominan Inggris
print(cm)

import statistics as st
from src.cot_synthesis.utils import read_jsonl

cot_rows = read_jsonl(SFT_DIR / "cot.jsonl")
lens = [len(r["messages"][1]["content"]) for r in cot_rows]

print(f"\nTRAIN-READY : {len(cot_rows)} pasang (cot + nocot)")
print(f"panjang CoT : median {int(st.median(lens))} char, p90 "
      f"{int(sorted(lens)[int(.9*len(lens))])}, maks {max(lens)}")
print(f"holdout     : {len(read_jsonl(EVAL_DIR / 'holdout.jsonl'))} soal")
print(f"dibuang krn bahasa Inggris : {cm.get('skipped_lang', 0)}")


In [ ]:
# ── 8b. GERBANG: himpunan soal cot.jsonl == nocot.jsonl ────────────────────
# Ini yang mengisolasi efek CoT di Skenario 3. Kalau gagal, JANGAN lanjut training --
# selisih akurasi antar-arm tidak lagi bisa diklaim sebagai efek CoT.
from src.cot_synthesis.verify_pairing import run as verify_pairing

pair = verify_pairing(SFT_DIR / "cot.jsonl", SFT_DIR / "nocot.jsonl")
for k, v in pair.items():
    if k != "masalah":
        print(f"{k:30} {v}")
assert pair["ok"], f"himpunan soal TIDAK identik: {pair['masalah']}"
print("\nOK: kedua file punya himpunan soal IDENTIK dan berpasangan per baris.")


In [ ]:
# ── 8c. Statistik panjang token (tokenizer Qwen) -> max_seq_length ─────────
# JANGAN pakai angka lama (p50 683 / p99 2001 / maks 3233) -- itu dari R1-Distill-7B pada
# data lama. Teacher 14B dengan thinking mode bisa jauh lebih panjang; #7 memakai angka ini.
from src.cot_synthesis.token_stats import run as token_stats, cetak as cetak_token

ts = token_stats(SFT_DIR / "cot.jsonl", tokenizer="Qwen/Qwen2.5-3B",
                 out_json=SFT_DIR / "token_stats.json")
cetak_token(ts)
cetak_token(token_stats(SFT_DIR / "nocot.jsonl", tokenizer="Qwen/Qwen2.5-3B"))

rekom = ts["max_seq_length_rekomendasi"]
print(f"\n>>> setel max_seq_length: {rekom} di src/training/configs/cot_3b.yaml "
      f"(sekarang 4096) -- {'SUDAH COCOK' if rekom == 4096 else 'PERLU DIUBAH'}")
print(">>> salin tabel di atas ke komentar issue #6; #7 memakai angka ini.")


In [ ]:
# ── 9. Paket hasil ──────────────────────────────────────────────────────────
import shutil

bundle = REPO / "outputs_cot_bundle"
bundle.mkdir(exist_ok=True)
for f in [SFT_DIR / "cot.jsonl", SFT_DIR / "nocot.jsonl", SFT_DIR / "token_stats.json",
          EVAL_DIR / "holdout.jsonl", DATA / "train_pool.jsonl",
          CORRECT_P, COT / "bakeoff_summary.json"]:
    if f.exists():
        shutil.copy(f, bundle / f.name)

# candidates_<teacher>.jsonl tidak ikut zip (puluhan GB) tapi WAJIB disimpan di disk:
# bahan mentah #9 dan #10. Cetak ukurannya supaya ketahuan kalau hilang.
if CAND_P.exists():
    print(f"kandidat mentah disimpan: {CAND_P}  ({CAND_P.stat().st_size/1e9:.2f} GB)")
else:
    print(f"PERINGATAN: {CAND_P} tidak ada -- #9/#10 harus generasi ulang")

shutil.make_archive(str(REPO / "cot_bundle"), "zip", bundle)
print("paket ->", REPO / "cot_bundle.zip")
print("isi:", [p.name for p in bundle.iterdir()])
